In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Import required packages

import numpy as np
import pandas as pd
import mlba
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# from imblearn.over_sampling import SMOTENC
from sklearn.preprocessing import (
    StandardScaler, 
    OneHotEncoder,  
    OrdinalEncoder
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.compose import ColumnTransformer
from sklearn import preprocessing
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, 
    cross_validate,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.base import clone
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.compose import ColumnTransformer


## Modeling (Decision Tree)

Predicting Credit Card Defaults

Mia Arnold - Katia Paredes - Harshali Gaikwad

Shirley Marcos School of Engineering, University of San Diego

ADS-505 Applied Data Science for Buisness

October 19th, 2026

## Table of Contents
1. [Data Preprocessing](#Data-Preprocessing)
2. [Modeling with Demographic Info](#Modeling-with-Demographic-Info)
3. [Modeling without Demographic Info](#Modeling-without-Demographic-Info)
4. [Model Performance](#Model-Performance)
5. [Model Selection](#Model-Selection)
6. [Discussion](#Discussion)

# Data Preprocessing

In [3]:
# load data

df = pd.read_csv('../data/credit_default_engineered.csv')
df_2 = df.drop(columns=['SEX','MARRIAGE','AGE'])

print(f'Shape of Data: {df.shape}')
print(f'Shape of Data w/o Demographics: {df_2.shape}')

Shape of Data: (30000, 41)
Shape of Data w/o Demographics: (30000, 38)


In [4]:
df.drop(columns= ['EDUCATION_label','MARRIAGE_label','DEFAULT_label', 'SEX_label'],inplace=True)
df.columns.to_list()

['LIMIT_BAL',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'AGE_GROUP',
 'UTIL_1',
 'UTIL_2',
 'UTIL_3',
 'UTIL_4',
 'UTIL_5',
 'UTIL_6',
 'UTIL_AVG',
 'UTIL_MAX',
 'UTIL_RECENT',
 'UTIL_CHANGE',
 'LIMIT_BAND',
 'HIGH_UTIL_MONTHS',
 'DEFAULT']

In [5]:
default_rate = df['DEFAULT'].mean()
print(f'baseline accuracy: {1 - default_rate}')

baseline accuracy: 0.7787999999999999


In [6]:
cat_cols = [
    'SEX',
    'EDUCATION',
    'MARRIAGE',
    'LIMIT_BAND',
    'AGE_GROUP'
]

for col in cat_cols:
    df[col] = df[col].astype('category') 


df.dtypes

LIMIT_BAL              int64
SEX                 category
EDUCATION           category
MARRIAGE            category
AGE                    int64
PAY_0                  int64
PAY_2                  int64
PAY_3                  int64
PAY_4                  int64
PAY_5                  int64
PAY_6                  int64
BILL_AMT1              int64
BILL_AMT2              int64
BILL_AMT3              int64
BILL_AMT4              int64
BILL_AMT5              int64
BILL_AMT6              int64
PAY_AMT1               int64
PAY_AMT2               int64
PAY_AMT3               int64
PAY_AMT4               int64
PAY_AMT5               int64
PAY_AMT6               int64
AGE_GROUP           category
UTIL_1               float64
UTIL_2               float64
UTIL_3               float64
UTIL_4               float64
UTIL_5               float64
UTIL_6               float64
UTIL_AVG             float64
UTIL_MAX             float64
UTIL_RECENT          float64
UTIL_CHANGE          float64
LIMIT_BAND    

In [7]:
# split 60/20/20 

X = df.drop(columns=['DEFAULT'])
y = df['DEFAULT']

# First split 60% train
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    train_size=.6,
    random_state=42,
    stratify=y,
)

# Second split 20% validation, 20% test
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    train_size=.5,
    random_state=42,
    stratify=y_temp,
)

print('Training:', X_train.shape)
print('Validation:', X_validation.shape)
print('Test:', X_test.shape)


print('Training purchase rate:')
print(y_train.mean())

print('\nValidation purchase rate:')
print(y_validation.mean())

print('\nTest purchase rate:')
print(y_test.mean())

Training: (18000, 36)
Validation: (6000, 36)
Test: (6000, 36)
Training purchase rate:
0.2212222222222222

Validation purchase rate:
0.22116666666666668

Test purchase rate:
0.22116666666666668


In [8]:
original_features = [
    'LIMIT_BAL',
    'SEX',
    'EDUCATION',
    'MARRIAGE',
    'AGE',
    'PAY_0',
    'PAY_2',
    'PAY_3',
    'PAY_4',
    'PAY_5',
    'PAY_6',
    'BILL_AMT1',
    'BILL_AMT2',
    'BILL_AMT3',
    'BILL_AMT4',
    'BILL_AMT5',
    'BILL_AMT6',
    'PAY_AMT1',
    'PAY_AMT2',
    'PAY_AMT3',
    'PAY_AMT4',
    'PAY_AMT5',
    'PAY_AMT6'
]



categorical_features = [
    'SEX',
    'MARRIAGE'
]
ordinal_features = [
    'EDUCATION'
]

numeric_features = [
    col for col in original_features
    if col not in categorical_features
]

# Modeling with Demographic Info and Original Features

In [9]:
# tree preprocessor

tree_preprocessor = ColumnTransformer(
    transformers = [
        (
            'numeric',
            'passthrough',
            numeric_features
        ),
        (
            'ordinal',
            OrdinalEncoder(
                categories=[['1','2','3','4']]
            ),
            ordinal_features
        ),
        (
           'category',
            OneHotEncoder(
                drop='first', 
                handle_unknown='ignore'
            ),
            categorical_features
        )
    ]
)      
       
 

In [10]:
# Tree model pipeline
tree_model = Pipeline(
    steps=[
        (
            'preprocessor',
            tree_preprocessor
        ),
        (
            'classifier',
            DecisionTreeClassifier(criterion='gini',random_state=42)
        )
    ]
)

In [11]:
# tree model
tree_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](36,)","['LIMIT_BAL','SEX','EDUCATION',...,'UTIL_CHANGE','LIMIT_BAND', 'HIGH_UTIL_MONTHS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,36
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``re

In [12]:
# predicted classes
tree_pred = tree_model.predict(X_validation)
# predicted probability 
tree_prob = tree_model.predict_proba(X_validation)[:,1]

In [13]:
print('Accuracy:', accuracy_score(y_validation, tree_pred))

print('Recall:', recall_score(y_validation, tree_pred))

print('Precision:', precision_score(y_validation, tree_pred))

print('F1 Score:', f1_score(y_validation, tree_pred))

print('ROC-AUC:', roc_auc_score(y_validation, tree_prob))

print('\nConfusion Matrix:')
print(confusion_matrix(y_validation, tree_pred))

Accuracy: 0.7235
Recall: 0.39261492087415223
Precision: 0.3791848617176128
F1 Score: 0.3857830433172899
ROC-AUC: 0.6055245134267936

Confusion Matrix:
[[3820  853]
 [ 806  521]]


### Decision Tree Tuning

In [14]:
# cross validation 
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [15]:

tree_parameter = {
    'classifier__max_depth': [3, 5, 7, 10],
    'classifier__min_samples_leaf': [10, 25, 50, 100]
}

tree_grid = GridSearchCV(
    estimator=tree_model,
    param_grid=tree_parameter,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

tree_grid.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [10, 25, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"

In [16]:
print(f'Best parameters: {tree_grid.best_params_}')
print(f'Best AV: {tree_grid.best_score_}')

Best parameters: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 100}
Best AV: 0.5317443490783306


In [17]:
# best tree from grid search
best_tree = tree_grid.best_estimator_

In [18]:
best_tree_pred = best_tree.predict(X_validation)
best_tree_proba = best_tree.predict_proba(X_validation)[:,1]

In [19]:
print('Accuracy:', accuracy_score(y_validation, best_tree_pred))

print('Recall:', recall_score(y_validation, best_tree_pred))

print('Precision:', precision_score(y_validation, best_tree_pred))

print('F1 Score:', f1_score(y_validation, best_tree_pred))

print('ROC-AUC:', roc_auc_score(y_validation, best_tree_pred))

print('\nConfusion Matrix:')
print(confusion_matrix(y_validation, best_tree_pred))

Accuracy: 0.8123333333333334
Recall: 0.32328560663149963
Precision: 0.6529680365296804
F1 Score: 0.4324596774193548
ROC-AUC: 0.6372473400159424

Confusion Matrix:
[[4445  228]
 [ 898  429]]


In [20]:
# fitted preprocessing step
preprocessor = best_tree.named_steps['preprocessor']

# feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

#  feature importance from the decision tree
importances = best_tree.named_steps[
    'classifier'
].feature_importances_

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance

,Feature,Importance
3,numeric__PAY_0,0.637450
4,numeric__PAY_2,0.120879
16,numeric__PAY_AMT2,0.040418
6,numeric__PAY_4,0.036811
0,numeric__LIMIT_BAL,0.024055
17,numeric__PAY_AMT3,0.021283
9,numeric__BILL_AMT1,0.018808
5,numeric__PAY_3,0.018521
7,numeric__PAY_5,0.016792
20,numeric__PAY_AMT6,0.015165


### Decision Tree with Weights

Using the parameters from best_tree and then add class weights. Given that the tree structure is already optimized, adding weights is the next logical step.

In [21]:
# clone the tuned pipeline (best tree)
weighted_best_tree = clone(best_tree)

# Add class weights to the decision tree inside the pipeline
weighted_best_tree.set_params(
    classifier__class_weight='balanced'
)

# Refit the complete pipeline
weighted_best_tree.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](36,)","['LIMIT_BAL','SEX','EDUCATION',...,'UTIL_CHANGE','LIMIT_BAND', 'HIGH_UTIL_MONTHS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,36
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``re

In [22]:
weighted_best_tree.named_steps[
    'classifier'
].get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': 10,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 100,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 42,
 'splitter': 'best'}

In [23]:
# predictions
weighted_tree_pred = weighted_best_tree.predict(X_validation)

# probabilites
weighted_tree_proba = weighted_best_tree.predict_proba(X_validation)[:, 1]

In [24]:
print('Accuracy:', accuracy_score(y_validation, weighted_tree_pred))

print('Recall:', recall_score(y_validation, weighted_tree_pred))

print('Precision:', precision_score(y_validation, weighted_tree_pred))

print('F1 Score:', f1_score(y_validation, weighted_tree_pred))

print('ROC-AUC:', roc_auc_score(y_validation, weighted_tree_proba))

print('\nConfusion Matrix:')
print(confusion_matrix(y_validation, weighted_tree_pred))

Accuracy: 0.7395
Recall: 0.5975885455915599
Precision: 0.43523600439077936
F1 Score: 0.5036519530009527
ROC-AUC: 0.7498288602081803

Confusion Matrix:
[[3644 1029]
 [ 534  793]]


In [25]:
comparison = pd.DataFrame({
    'Model': [
        'Decision Tree',
        'Best Tree',
        'Best Tree + Balanced Weights'
    ],
    'Accuracy': [
        accuracy_score(y_validation, tree_pred),
        accuracy_score(y_validation, best_tree_pred),
        accuracy_score(y_validation, weighted_tree_pred)
    ],
    'Recall': [
        recall_score(y_validation, tree_pred),
        recall_score(y_validation, best_tree_pred),
        recall_score(y_validation, weighted_tree_pred)
    ],
    'Precision': [
        precision_score(y_validation, tree_pred),
        precision_score(y_validation, best_tree_pred),
        precision_score(y_validation, weighted_tree_pred)
    ],
    'F1': [
        f1_score(y_validation, tree_pred),
        f1_score(y_validation, best_tree_pred),
        f1_score(y_validation, weighted_tree_pred)
    ],
    'ROC-AUC': [
        roc_auc_score(y_validation, tree_prob),
        roc_auc_score(y_validation, best_tree_proba),
        roc_auc_score(y_validation, weighted_tree_proba)
    ]
})

comparison.round(3)

,Model,Accuracy,Recall,Precision,F1,ROC-AUC
0,Decision Tree,0.724,0.393,0.379,0.386,0.606
1,Best Tree,0.812,0.323,0.653,0.432,0.759
2,Best Tree + Balanced Weights,0.740,0.598,0.435,0.504,0.750


In [26]:
print('Best Tree')
print(
    confusion_matrix(
        y_validation,
        best_tree_pred
    )
)

print('\nBest Tree + Balanced Weights')
print(
    confusion_matrix(
        y_validation,
        weighted_tree_pred
    )
)

Best Tree
[[4445  228]
 [ 898  429]]

Best Tree + Balanced Weights
[[3644 1029]
 [ 534  793]]


In [27]:
def get_metrics(model, X, y):
    # Predict classes
    y_pred = model.predict(X)

    # Predict probability of default
    y_prob = model.predict_proba(X)[:, 1]

    # Return evaluation metrics
    return {
        'Accuracy': accuracy_score(y, y_pred),
        'Recall': recall_score(y, y_pred),
        'Precision': precision_score(y, y_pred),
        'F1': f1_score(y, y_pred),
        'ROC-AUC': roc_auc_score(y, y_prob)
    }

results = []

for model_name, model in [
    ('Best Tree', best_tree),
    (
        'Best Tree + Balanced Weights',
        weighted_best_tree
    )
]:
    for dataset_name, X_data, y_data in [
        ('Training', X_train, y_train),
        ('Validation', X_validation, y_validation)
    ]:
        metrics = get_metrics(
            model,
            X_data,
            y_data
        )

        results.append({
            'Model': model_name,
            'Dataset': dataset_name,
            **metrics
        })

train_valid_comparison = pd.DataFrame(results)

train_valid_comparison.round(3)

,Model,Dataset,Accuracy,Recall,Precision,F1,ROC-AUC
0,Best Tree,Training,0.825,0.366,0.701,0.481,0.800
1,Best Tree,Validation,0.812,0.323,0.653,0.432,0.759
2,Best Tree + Balanced Weights,Training,0.766,0.653,0.479,0.552,0.803
3,Best Tree + Balanced Weights,Validation,0.740,0.598,0.435,0.504,0.750


**Training and Validation Performance:** Both tuned decision-tree models perform consistently across the training and validation datasets, with slightly lower validation metrics and no evidence of significant overfitting. Applying balanced class weights increased validation recall from 0.323 to 0.598 and improved the F1 score from 0.432 to 0.504, indicating that the weighted model detects a greater share of customers who later default. However, it also produces lower precision and accuracy, indicating more false-positive classifications. ROC-AUC slightly decreased from 0.759 to 0.750. 

From a business perspective, the weighted model may be preferable when the cost of failing to identify a future defaulter exceeds the cost of conducting additional outreach to customers who ultimately do not default. However, depending on business priorities, the Best Tree could be the optimal choice since it flags fewer customers unnecessarily.

# Modeling with Engineered Features

In [28]:
engineered_features = [
    'UTIL_AVG',
    'UTIL_MAX',
    'UTIL_RECENT',
    'UTIL_CHANGE',
    'UTIL_1',
    'UTIL_2',
    'UTIL_3',
    'UTIL_4',
    'UTIL_5',
    'UTIL_6'
]
# Count months in which utilization was at least 75%
util_cols = [
    'UTIL_1',
    'UTIL_2',
    'UTIL_3',
    'UTIL_4',
    'UTIL_5',
    'UTIL_6'
]

df['HIGH_UTIL_MONTHS'] = (df[util_cols] >= 0.75).sum(axis=1)

engineered_features = [
    'UTIL_AVG',
    'UTIL_MAX',
    'UTIL_CHANGE',
    'HIGH_UTIL_MONTHS'
]

In [29]:
engineered_model_features = (
    original_features
    + engineered_features
)

X_eng = df[engineered_model_features]

y = df['DEFAULT']

In [30]:
X_eng_train, X_eng_temp, y_eng_train, y_eng_temp = (
    train_test_split(
        X_eng,
        y,
        test_size=0.40,
        stratify=y,
        random_state=1
    )
)

X_eng_valid, X_eng_test, y_eng_valid, y_eng_test = (
    train_test_split(
        X_eng_temp,
        y_eng_temp,
        test_size=0.50,
        stratify=y_eng_temp,
        random_state=1
    )
)
print(
    'Training:',
    X_eng_train.shape
)

print(
    'Validation:',
    X_eng_valid.shape
)

print(
    'Holdout:',
    X_eng_test.shape
)

Training: (18000, 27)
Validation: (6000, 27)
Holdout: (6000, 27)


In [31]:
categorical_features = [
    'SEX',
    'MARRIAGE'
]
ordinal_features = [
    'EDUCATION'
]

engineered_numeric_features = [
    col for col in original_features
    if col not in categorical_features
]

In [33]:
# tree preprocessor

tree_eng_preprocessor = ColumnTransformer(
    transformers = [
        (
            'numeric',
            'passthrough',
            numeric_features
        ),
        (
            'ordinal',
            OrdinalEncoder(
                categories=[['1','2','3','4']]
            ),
            ordinal_features
        ),
        (
           'category',
            OneHotEncoder(
                drop='first', 
                handle_unknown='ignore'
            ),
            categorical_features
        )
    ]
)      
       

In [34]:
# engineered pipeline
engineered_tree = Pipeline(
    steps=[
        (
            'preprocessor',
            tree_eng_preprocessor
        ),
        (
            'classifier',
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

In [35]:
tree_eng_params = {
    'classifier__max_depth': [3, 5, 7, 10],
    'classifier__min_samples_leaf': [10, 25, 50, 100]
}

In [36]:
engineered_tree_grid = GridSearchCV(
    estimator=engineered_tree,
    param_grid=tree_eng_params,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

engineered_tree_grid.fit(X_eng_train, y_eng_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [10, 25, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls

In [37]:
print('Best parameters:', engineered_tree_grid.best_params_)

print('Best CV ROC-AUC:', engineered_tree_grid.best_score_)

Best parameters: {'classifier__max_depth': 7, 'classifier__min_samples_leaf': 100}
Best CV ROC-AUC: 0.7634160389432889


In [38]:
best_engineered_tree = (engineered_tree_grid.best_estimator_)

In [39]:
weighted_engineered_tree = clone(best_engineered_tree)

# class weights
weighted_engineered_tree.set_params(classifier__class_weight='balanced')

weighted_engineered_tree.fit(X_eng_train,y_eng_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](27,)","['LIMIT_BAL','SEX','EDUCATION',...,'UTIL_MAX','UTIL_CHANGE', 'HIGH_UTIL_MONTHS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,27
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``rema

In [40]:
def get_metrics(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        'Accuracy': accuracy_score(y,y_pred),
        'Recall': recall_score(y,y_pred),
        'Precision': precision_score(y,y_pred),
        'F1': f1_score(y,y_pred),
        'ROC-AUC': roc_auc_score(y,y_prob)}

In [41]:
eng_results = []

for model_name, model in [
    (
        'Best Tree + Engineered Features',
        best_engineered_tree
    ),
    (
        'Weighted Tree + Engineered Features',
        weighted_engineered_tree
    )
]:
    for dataset_name, X_data, y_data in [
        (
            'Training',
            X_eng_train,
            y_eng_train
        ),
        (
            'Validation',
            X_eng_valid,
            y_eng_valid
        )
    ]:
        metrics = get_metrics(
            model,
            X_data,
            y_data
        )

        eng_results.append({
            'Model': model_name,
            'Dataset': dataset_name,
            **metrics
        })

eng_comparison = pd.DataFrame(
    eng_results
)

eng_comparison.round(3)

,Model,Dataset,Accuracy,Recall,Precision,F1,ROC-AUC
0,Best Tree + Engineered Features,Training,0.823,0.393,0.672,0.496,0.780
1,Best Tree + Engineered Features,Validation,0.810,0.365,0.619,0.460,0.764
2,Weighted Tree + Engineered Features,Training,0.738,0.681,0.440,0.535,0.791
3,Weighted Tree + Engineered Features,Validation,0.720,0.639,0.414,0.502,0.762


**Impact of Feature Engineering:** For the tuned decision tree, the engineered features increased validation recall from 0.323 to 0.365, F1 from 0.432 to 0.460, and ROC-AUC from 0.759 to 0.764. For the class-weighted tree, recall increased from 0.598 to 0.639 and ROC-AUC from 0.750 to 0.762, although precision declined from 0.435 to 0.414. Training and validation performance remained similar, suggesting that the additional features did not cause significant overfitting. Overall, the results indicate that the engineered utilization features provide modest additional predictive information, particularly for identifying a larger share of customers who default.